# Package

In [1]:
# ============================================================
# 1) Core Python & Paths
# ============================================================
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from dataclasses import dataclass
from typing import Callable, Dict, Any, Optional, List, Iterable

from dateutil.relativedelta import relativedelta

# Project root (notebook dans /notebooks)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 2) Scientific Stack (NumPy / Pandas)
# ============================================================
import numpy as np
import pandas as pd


# ============================================================
# 3) Machine Learning & Models
# ============================================================
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import (
    GridSearchCV,
    ParameterGrid,
    ParameterSampler
)

from lightgbm import LGBMRegressor


# ============================================================
# 4) Forecasting (Nixtla MLForecast)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals


# ============================================================
# 5) Data & Project Utilities
# ============================================================
from utils import (
    load_wide_from_feast,
    apply_exog_lags,
    build_unrate_exog_dataset,
    ModelSpec,
    run_backtesting_generic,
)

# ============================================================
# 6) Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

# ============================================================
# 7) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display


# ============================================================
# 8) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient

# Parameters

## Data parameters

In [2]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

LAG = 24

feast_data_name = "stationary_value:value"

## Model

In [3]:
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"
EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")
SEED = 42
min_train_n = 36

# Load Data

In [4]:
# ============================================================
# 1) WIDE dataset depuis Feast
# ============================================================

df = load_wide_from_feast(
    feast_data_name,
    series_ids,
    start=START,
    end=END
)


# ============================================================
# 2) Convert WIDE -> LONG
# ============================================================

df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)


# ============================================================
# 3) Build dataset MLForecast
# ============================================================

df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)


# ============================================================
# 4) LAG automatique des variables exogènes
# ============================================================

ts_lr, exog_cols = apply_exog_lags(
    ts_lr,
    exog_cols=exog_cols,
    lags=LAG,            
    drop_original_exog=True,
    dropna=True
)

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:148: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  idx = idx.to_period("M").to_timestamp(how="start").normalize()


In [5]:
# ============================================================
# 5) Vérifications
# ============================================================

print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag):", ts_lr.shape)
print("Exog cols:", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag): (764, 13)
Exog cols: ['BUSLOANS_lag24', 'CPIAUCSL_lag24', 'DPCERA3M086SBEA_lag24', 'INDPRO_lag24', 'M2SL_lag24', 'OILPRICEX_lag24', 'RPI_lag24', 'SP500_lag24', 'TB3MS_lag24', 'USREC_lag24']

Preview:
   unique_id         ds    y  BUSLOANS_lag24  CPIAUCSL_lag24  \
0     UNRATE 1962-01-01 -0.8        0.011578       -0.006156   
1     UNRATE 1962-02-01 -1.4        0.011905       -0.003767   
2     UNRATE 1962-03-01 -1.3       -0.008356       -0.005455   
3     UNRATE 1962-04-01 -1.4       -0.009098        0.005090   
4     UNRATE 1962-05-01 -1.6       -0.000359        0.003383   
5     UNRATE 1962-06-01 -1.4        0.014620        0.006777   
6     UNRATE 1962-07-01 -1.6       -0.000611       -0.005433   
7     UNRATE 1962-08-01 -0.9       -0.016888       -0.004074   
8     UNRATE 1962-09-01 -1.1       -0.031830       -0.006777   
9     UNRATE 1962-10-01 -1.1       -0.018393

# Model settings

In [6]:
# ============================================================
# Builders modèles
# ============================================================

def build_lr(freq, params):
    return MLForecast(models={"LR": LinearRegression()}, freq=freq, lags=[], date_features=[])

def build_ridge(freq, params):
    alpha = params.get("alpha", 1.0)
    return MLForecast(models={"RIDGE": Ridge(alpha=alpha)}, freq=freq, lags=[], date_features=[])

LGBM_BASE = dict(random_state=0, n_jobs=-1, verbosity=-1, objective="regression", metric="mae")

def build_lgbm(freq, params):
    p = dict(LGBM_BASE)
    p.update(params)
    return MLForecast(models={"LGBM": LGBMRegressor(**p)}, freq=freq, lags=[], date_features=[])

# ============================================================
# Model Specs (LR + RIDGE + LGBM)
# ============================================================

MODEL_SPECS = [
    ModelSpec(
        name="LR",
        build_mlf=build_lr,
        pred_col="LR",
        tunable=False,
        fixed_params={},
    ),
    ModelSpec(
        name="RIDGE",
        build_mlf=build_ridge,
        pred_col="RIDGE",
        tunable=True,
        param_space={"alpha": np.logspace(-4, 4, 30)},  # ✅ corrigé
        search="grid",
        tune_every_months=36,
    ),
    ModelSpec(
        name="LGBM",
        build_mlf=build_lgbm,
        pred_col="LGBM",
        tunable=True,
        param_space={
            "subsample":        [0.05, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0],
            "colsample_bytree": [.2, .3, .4, .5, .6, .7, 1.0],
            "num_leaves":       [2, 3, 4, 5, 8, 10, 20, 40, 70, 100],
            "n_estimators":     [5, 10, 20, 30, 40, 50, 75, 100],
            "max_depth":        [1, 2, 3, 5, 8, 15, -1],
            "reg_alpha":        [0, .1, 1, 2, 7, 10, 50, 100],
            "reg_lambda":       [0, .1, 1, 10, 20, 50, 100],
            "min_child_samples":[5, 10, 15],
            "min_split_gain":   [0.0, 0.01, 0.05],
        },  # ✅ accolade corrigée
        search="random",
        n_iter=12,
        tune_every_months=36,
    ),
]

In [7]:
# 2) Run LR + RIDGE (avec retrain/tuning par blocs pour Ridge)

bkt_models_final, meta_models = run_backtesting_generic(
    ts=ts_lr,
    model_specs=MODEL_SPECS, 
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    seed=SEED,
    min_train_n=min_train_n,
)

print("✅ rows:", len(bkt_models_final))
print("✅ columns (head):", bkt_models_final.columns.tolist()[:25])

# vérifier blocs Ridge
print("✅ Ridge blocks:", len(meta_models["metas"]["RIDGE"].get("params_history", [])))

# vérifier blocs LGBM
print("✅ LGBM blocks:", len(meta_models["metas"]["LGBM"].get("params_history", [])))

bkt_models_final.head()

✅ rows: 428
✅ columns (head): ['unique_id', 'ds', 'cutoff', 'y', 'LR', 'LR-lo-95', 'LR-hi-95', 'LR_tune_block', 'LR_tune_mae', 'RIDGE', 'RIDGE-lo-95', 'RIDGE-hi-95', 'RIDGE_tune_block', 'RIDGE_tune_mae', 'LGBM', 'LGBM-lo-95', 'LGBM-hi-95', 'LGBM_tune_block', 'LGBM_tune_mae']
✅ Ridge blocks: 12
✅ LGBM blocks: 12


,unique_id,ds,cutoff,y,LR,LR-lo-95,LR-hi-95,LR_tune_block,LR_tune_mae,RIDGE,RIDGE-lo-95,RIDGE-hi-95,RIDGE_tune_block,RIDGE_tune_mae,LGBM,LGBM-lo-95,LGBM-hi-95,LGBM_tune_block,LGBM_tune_mae
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.143011,-1.251573,0.965552,1,NaN,-0.100528,-0.954433,0.753377,1,0.665744,-0.135347,-0.724123,0.453429,1,0.459511
1,UNRATE,1990-02-01,1990-01-01,0.1,-0.006611,-1.365453,1.352230,1,NaN,-0.032936,-0.895307,0.829435,1,0.665744,-0.175118,-0.746138,0.395902,1,0.459511
2,UNRATE,1990-03-01,1990-02-01,0.2,-0.228764,-1.941860,1.484331,1,NaN,-0.041391,-0.852942,0.770159,1,0.665744,-0.590978,-1.510310,0.328354,1,0.459511
3,UNRATE,1990-04-01,1990-03-01,0.2,0.098008,-1.138987,1.335004,1,NaN,-0.002270,-0.858847,0.854307,1,0.665744,-0.386869,-1.173660,0.399922,1,0.459511
4,UNRATE,1990-05-01,1990-04-01,0.2,0.054069,-0.791636,0.899775,1,NaN,0.111818,-0.618965,0.842601,1,0.665744,-0.029279,-0.469799,0.411241,1,0.459511
